# 步骤 05 · 包络谱 —— 项目的高潮

**这一节的产出：图 5a（包络是怎么求出来的）+ 图 5b（四条记录的包络谱，带理论频率和谐波标注）。**

## 走到这里，路已经铺好了

| 步骤 | 结论 |
|---|---|
| 01 波形 | 外圈能用眼睛数出 21 簇，内圈糊、滚珠几乎看不出 |
| 02 统计量 | 能说"有病"，说不出"什么病"，而且原理上不可能 |
| 03 直接 FFT | 故障频率处有峰，但**健康记录也有** —— 会误判 |
| 04 时频图 | 能量在 2–4 kHz **周期性闪烁**；把闪烁做 FFT，外圈 601、内圈 222 |

步骤 04 其实已经把答案做出来了。这一节做的是**用正确的工具重做一遍**，并且把最后一个问题暴露出来。

## 这一节的三行核心代码

```python
xf  = filtfilt(b, a, x)      # 1. 带通滤波，只留高频共振那一段
env = np.abs(hilbert(xf))    # 2. 求包络（外轮廓）
A   = np.fft.rfft(env - env.mean())   # 3. 对包络做 FFT
```

**就这三行。** 1970 年代就成熟了的技术。这一节剩下的全部篇幅，是为了让你明白**每一行为什么在那儿**。

---
## 1. 包络是什么

回到图 4a 那张时频图。外圈那一行是一条条亮纹：撞一下 → 亮 → 衰减 → 再撞一下。

如果把那个信号画成波形，它长这样：一个 **3000 Hz 的快速振荡**，外面被一个 **105.9 Hz 的慢速起伏**包着。

**包络（envelope）就是那个"慢速起伏"** —— 把快速振荡的峰顶连起来画出的那条外轮廓线。

| | 频率 | 是什么 |
|---|---|---|
| 载波（carrier） | 3000 Hz | 轴承座被敲响的固有共振 |
| 包络（envelope） | 105.9 Hz | **撞击的节奏 ← 我们要的** |

步骤 03 直接做 FFT，看到的是**载波**（2–4 kHz 那一大片）。
**包络分析做的事，就是把载波扔掉，只留下包络，然后对包络做 FFT。**

> 一个类比：远处传来鼓声。**鼓皮的音高**（载波）取决于鼓多大、皮多紧；**鼓点的节奏**（包络）取决于鼓手。你想知道这首曲子的节拍，不该去分析鼓的音高 —— 该去数鼓点。
>
> 步骤 03 分析了音高。这一节数鼓点。

---
## 2. 希尔伯特变换：怎么把外轮廓求出来

"把峰顶连起来"听着简单，真要用代码干却很别扭 —— 找局部最大值？最大值之间怎么插值？噪声造出假峰怎么办？

**希尔伯特变换给了一个精确、无歧义的答案。**

### 直观的解释

想象一个匀速转动的向量（箭头），长度会变，转速很快。

- 你的信号 $x(t)$ = 这个向量在**水平方向的投影**
- 希尔伯特变换给出的是它在**竖直方向的投影**
- 两个投影凑成一对，就能算出**向量本身的长度**：$\sqrt{x^2 + \hat{x}^2}$

**那个长度就是包络。**

单看水平投影，你只看到它在正负之间摆动，看不出向量多长。**补上竖直投影，长度立刻确定。**

数学上，`hilbert(x)` 返回的是**解析信号** $x(t) + i\,\hat{x}(t)$（一个复数数组），`np.abs()` 取它的模长 —— 就是向量长度 —— 于是包络出来了。

> 为什么这样求出的包络是"对的"？因为对于 $A(t)\sin(2\pi f_c t)$ 这种"慢变幅度 × 快速振荡"的信号，希尔伯特变换恰好把 $\sin$ 转成 $\cos$，于是
> $$\sqrt{(A\sin)^2 + (A\cos)^2} = A$$
> 载波被干净地消掉，只剩幅度 $A(t)$。**前提是"慢变"和"快速"要分得开** —— 这就是下一段为什么必须先滤波。

### 为什么必须先带通滤波

希尔伯特变换只有在信号是**窄带**（能量集中在一个频段）时才给出有意义的包络。

原始信号里什么都有：低频的机器谱线、2–4 kHz 的共振、各种噪声。直接求包络，得到的是这一锅东西共同的外轮廓，没有物理意义。

**所以先用带通滤波器把 2000–4000 Hz 单独拿出来，别的全扔掉。** 扔掉的包括步骤 03 里那些会骗人的机器谱线 —— 它们在几百赫兹，一滤就没了。

> **这就是为什么包络谱能躲开步骤 03 的陷阱**：那根冒充 BPFI 的 159.5 Hz 机器谱线，在带通滤波这一步就被扔掉了，根本进不来。

---
## 3. 准备

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, hilbert

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import cwru_io

DATA = ROOT / "data"
FIGURES = ROOT / "figures"
FS = cwru_io.FS

normal = cwru_io.load_baseline(DATA / "normal_1hp_98.mat",    name="Healthy")
outer  = cwru_io.load(DATA / "OR007at6_1hp_131.mat", name="Outer race")
inner  = cwru_io.load(DATA / "IR007_1hp_106.mat",    name="Inner race")
ball   = cwru_io.load(DATA / "B007_1hp_119.mat",     name="Ball")

signals = [normal, outer, inner, ball]
N = min(len(s.x) for s in signals)

TARGET = {"Healthy": "BPFO", "Outer race": "BPFO",
          "Inner race": "BPFI", "Ball": "BSF"}
COLORS = {"Healthy": "#256049", "Outer race": "#8E3320",
          "Inner race": "#1B4F8F", "Ball": "#9A5F0A"}

BAND = (2000, 4000)          # 共振带 —— 这个选择是步骤 06 的主题
print(f"共振带暂定 {BAND[0]}-{BAND[1]} Hz（凭什么是这一段？步骤 06 回答）")

---
## 4. 三行代码，和一次已知答案的验证

In [ ]:
def envelope(x, band=BAND, fs=FS, order=4):
    # 带通滤波后求希尔伯特包络。
    nyq = fs / 2
    b, a = butter(order, [band[0] / nyq, band[1] / nyq], btype="band")
    xf = filtfilt(b, a, x)            # filtfilt: 正反各滤一次 -> 零相位
    return np.abs(hilbert(xf))


def envelope_spectrum(x, band=BAND, fs=FS):
    env = envelope(x, band, fs)
    env = env - env.mean()            # 去直流，否则 0 Hz 一根大峰（见第 8 节）
    n = len(env)
    f = np.fft.rfftfreq(n, 1 / fs)
    A = np.abs(np.fft.rfft(env * np.hanning(n))) / n * 4
    return f, A


def peak_ratio(f, A, f0, tol=2.5):
    near = np.abs(f - f0) <= tol
    bg = (np.abs(f - f0) > 10) & (np.abs(f - f0) < 60)
    return A[near].max() / np.median(A[bg]), f[near][np.argmax(A[near])]

`filtfilt` 而不是 `lfilter`：它把信号**正着滤一遍、倒过来再滤一遍**。

为什么要这么麻烦？因为任何滤波器都会让信号**延迟**，而且不同频率延迟的量还不一样（相位失真）。正反各滤一次，两次的延迟正好抵消 —— 这叫**零相位滤波**。

**对我们来说这是必须的**：我们要测的是"撞击在什么时刻发生"的节奏，如果不同频率成分被错开不同的时间，节奏就被搅乱了。

> 代价是 `filtfilt` 只能离线用（需要看到整段信号），实时系统里用不了。

### 现在验证它

**造一个我们完全知道答案的信号**：每隔 1/105.9 秒来一次撞击，每次撞击激起 3000 Hz 的衰减振荡，再加上 3 倍于信号强度的噪声。

这就是我们对轴承故障的物理模型。**如果方法是对的，它必须从这个信号里挖出 105.9 Hz。**

In [ ]:
def synthetic_bearing(f_impact=105.9, f_res=3000.0, tau=0.0008,
                      noise_ratio=3.0, duration=10.0, fs=FS, seed=0):
    # 造一个人工轴承信号：周期性冲击 -> 激起衰减的高频共振 -> 埋进噪声
    n = int(fs * duration)
    x = np.zeros(n)
    for k in range(int(duration * f_impact)):
        i0 = int(k / f_impact * fs)
        if i0 >= n:
            break
        L = min(int(6 * tau * fs), n - i0)
        tt = np.arange(L) / fs
        x[i0:i0 + L] += np.exp(-tt / tau) * np.sin(2 * np.pi * f_res * tt)
    rng = np.random.default_rng(seed)
    return x + rng.normal(0, noise_ratio * x.std(), n)


syn = synthetic_bearing()

# 对照 1：直接对原信号做 FFT（步骤 03 的做法）
n = len(syn)
f_raw = np.fft.rfftfreq(n, 1 / FS)
A_raw = np.abs(np.fft.rfft((syn - syn.mean()) * np.hanning(n))) / n * 4
r_raw, _ = peak_ratio(f_raw, A_raw, 105.9)

# 对照 2：包络谱
f_env, A_env = envelope_spectrum(syn)
r_env, fp = peak_ratio(f_env, A_env, 105.9)

print("人工信号（真实答案 = 105.9 Hz，由构造保证）\n")
print(f"  直接 FFT   峰背比 {r_raw:7.2f}   <- 埋掉了")
print(f"  包络谱     峰背比 {r_env:7.2f}   <- 挖出来了，峰在 {fp:.2f} Hz")
print()
for k in (2, 3):
    print(f"    {k} x 105.9 = {k*105.9:6.1f} Hz   峰背比 {peak_ratio(f_env, A_env, k*105.9)[0]:6.1f}")

**同一个信号、同一个频率：直接 FFT 是 3.0（等于没有），包络谱是 31.8，峰精确落在 105.90 Hz。**

这个验证的价值在于：信号是我们**自己造的**，105.9 Hz 是构造时放进去的，不存在"碰巧"的可能。方法确实在做我们以为它在做的事。

> 这是这个项目里第三次"先用已知答案验证再上真实数据"（前两次是步骤 02 的高斯噪声峭度、步骤 03 的正弦波幅值）。**养成这个习惯，比记住任何一个公式都有用。**

---
## 5. 图 5a：把三步画出来

拿外圈故障的 40 毫秒，看每一步做了什么。

In [ ]:
T0, T_LEN = 0.10, 0.04
i0, i1 = int(T0 * FS), int((T0 + T_LEN) * FS)
t = np.arange(i0, i1) / FS

x_raw = outer.x[i0:i1]
nyq = FS / 2
b, a = butter(4, [BAND[0] / nyq, BAND[1] / nyq], btype="band")
x_filt = filtfilt(b, a, outer.x[:N])[i0:i1]
env_full = envelope(outer.x[:N])[i0:i1]

fig5a, axes = plt.subplots(3, 1, figsize=(11, 7), sharex=True)

axes[0].plot(t, x_raw, linewidth=0.7, color="#5B6773")
axes[0].set_title("1. Raw signal  -  everything at once", fontsize=10, loc="left")

axes[1].plot(t, x_filt, linewidth=0.7, color="#8E3320")
axes[1].set_title(f"2. Band-pass {BAND[0]}-{BAND[1]} Hz  -  only the resonance left",
                  fontsize=10, loc="left")

axes[2].plot(t, x_filt, linewidth=0.6, color="#C9A79B")
axes[2].plot(t, env_full, linewidth=1.6, color="#8E3320")
axes[2].plot(t, -env_full, linewidth=1.6, color="#8E3320", alpha=0.45)
axes[2].set_title("3. Hilbert envelope  -  the outline, i.e. the rhythm",
                  fontsize=10, loc="left")

# 画一把"理论间距的梳子"：齿距由 BPFO 定死，整把梳子对齐到窗内最大的那个冲击。
# 检验的是【间距】对不对，不是绝对时刻 —— 冲击的起始相位取决于开始录数据时
# 滚珠转到了哪儿，理论无从预测。
bpfo = outer.fault_freqs()["BPFO"]
anchor = t[np.argmax(env_full)]
for k in range(-8, 9):
    tk = anchor + k / bpfo
    if t[0] <= tk <= t[-1]:
        for ax in axes:
            ax.axvline(tk, color="#1B4F8F", linestyle=":", linewidth=0.9, alpha=0.6)

for ax in axes:
    ax.set_ylabel("Accel. (g)")
    ax.grid(alpha=0.2)
axes[-1].set_xlabel("Time (s)")
fig5a.suptitle("Fig. 5a  Building the envelope, outer race, 40 ms\n"
               f"dotted comb: spacing fixed at BPFO = {bpfo:.1f} Hz "
               f"({1000/bpfo:.1f} ms), anchored on the largest impact",
               fontsize=11)
fig5a.tight_layout()
plt.show()

### 读图 5a

**第 1 行（原始信号）**：乱。有冲击，但混着别的东西。

**第 2 行（带通之后）**：干净多了 —— 只剩 2–4 kHz 的共振振荡，一簇一簇。低频的机器谱线**已经不在了**，步骤 03 那个 159.5 Hz 的陷阱在这一步就被排除掉了。

**第 3 行（包络）**：粗线就是包络，它把每一簇振荡"套"了起来。**现在每一次撞击变成了一个清清楚楚的鼓包。**

**蓝色竖点线是一把"梳子"：齿距被定死成 9.44 ms（= 1/BPFO），整把梳子对齐到窗内最大的那个鼓包。**

这里检验的是**间距**，不是绝对时刻 —— 第一次撞击发生在哪一瞬间，取决于开始录数据时滚珠正好转到哪儿，理论无从预测，也没必要预测。

**梳子锚好之后，看后面几个鼓包是不是每次都落在齿上。** 实测相邻鼓包间隔约 9.3 / 9.4 / 9.5 ms，理论 9.44 ms。

> 这一点值得单独说：**做验证时要分清"哪部分是理论该管的、哪部分不是"。** 间距由物理决定，必须对得上；相位由初始条件决定，对不上很正常。我第一版画图时把梳子从 t=0 起算，结果齿和鼓包整体错开了半个周期 —— 图看起来像是"理论错了"，其实只是我拿理论去要求了一个它管不着的东西。

这是你第三次验证同一件事（图 1 数簇、图 4a 数竖纹、现在包络鼓包的间距）。三种不同的方法，同一个答案。

---
## 6. 图 5b：包络谱

In [ ]:
espec = {s.name: envelope_spectrum(s.x[:N]) for s in signals}

LINES = [("BPFO", "#8E3320"), ("BPFI", "#1B4F8F"), ("BSF", "#9A5F0A")]

fig5b, axes = plt.subplots(4, 1, figsize=(11, 9), sharex=True)

for ax, s in zip(axes, signals):
    f, A = espec[s.name]
    m = f <= 500
    ax.plot(f[m], A[m] / A[m].max(), linewidth=0.8, color=COLORS[s.name])

    ff = s.fault_freqs()
    for k, c in LINES:
        ax.axvline(ff[k], color=c, linestyle="--", linewidth=1.1, alpha=0.8)

    # 这条记录自己那个频率的 2/3/4 倍频，用小三角标出来
    f0 = ff[TARGET[s.name]]
    for k in (2, 3, 4):
        if k * f0 <= 500:
            ax.plot(k * f0, 1.02, marker="v", markersize=5,
                    color=COLORS[s.name], clip_on=False)

    ax.set_ylabel("Norm. amp.")
    ax.set_ylim(0, 1.1)
    ax.grid(alpha=0.25)
    ax.text(0.011, 0.80, s.name, transform=ax.transAxes,
            fontsize=10, fontweight="bold")

ff = signals[0].fault_freqs()
for k, c in LINES:
    axes[0].text(ff[k], 1.20, f"{k}\n{ff[k]:.0f}", color=c, fontsize=8,
                 ha="center", va="bottom", linespacing=1.15)

axes[-1].set_xlabel("Frequency (Hz)")
axes[-1].set_xlim(0, 500)
axes[0].set_title(f"Fig. 5b  Envelope spectra, band {BAND[0]}-{BAND[1]} Hz\n"
                  "dashed: theoretical fault frequencies; "
                  "triangles: harmonics of this record's own frequency",
                  fontsize=11, pad=40)
fig5b.tight_layout()
plt.show()

In [ ]:
print("=== 每条记录在【它自己的】理论频率上 ===\n")
print(f"{'记录':<13}{'查':<7}{'理论 Hz':>10}{'峰背比':>10}{'实测 Hz':>10}{'误差':>9}")
print("-" * 60)
for s in signals:
    key = TARGET[s.name]
    f0 = s.fault_freqs()[key]
    f, A = espec[s.name]
    r, fp = peak_ratio(f, A, f0)
    print(f"{s.name:<13}{key:<7}{f0:>10.1f}{r:>10.1f}{fp:>10.1f}{(fp-f0)/f0:>8.2%}")

print("\n\n=== 交叉检验：对角线必须最亮 ===\n")
print(f"{'':<13}{'BPFO':>10}{'BPFI':>10}{'BSF':>10}")
print("-" * 45)
for s in signals:
    f, A = espec[s.name]
    ff = s.fault_freqs()
    vals = [peak_ratio(f, A, ff[k])[0] for k in ("BPFO", "BPFI", "BSF")]
    marks = ["  <<<" if k == TARGET[s.name] else "" for k in ("BPFO", "BPFI", "BSF")]
    print(f"{s.name:<13}" + "".join(f"{v:>10.1f}" for v in vals)
          + ("   <- 对角线" if s.name != "Healthy" else ""))

print("\n\n=== 谐波列：真信号该有 2x 3x 4x ===\n")
for s in signals[1:]:
    key = TARGET[s.name]
    f0 = s.fault_freqs()[key]
    f, A = espec[s.name]
    parts = []
    for k in range(1, 6):
        if k * f0 <= 600:
            parts.append(f"{k}x {peak_ratio(f, A, k*f0)[0]:6.1f}")
    print(f"  {s.name:<12}({key})  " + "   ".join(parts))

---
## 7. 结果

### 外圈：635，谐波成列

峰背比 **635**，实测 106.3 Hz vs 理论 105.9 Hz，**误差 0.4%**。而且 2×、3×、4×、5× 倍频全部高度突出。

**一整列谐波是比单根峰强得多的证据。** 噪声不会凑巧在五个整数倍位置同时冒峰。

### 内圈：237，同样干净

实测 159.5 vs 理论 159.9，误差 −0.25%。交叉检验那一行：BPFO 处 2.9、BSF 处 4.1、**BPFI 处 236.9**。

**只在理论预测的那一处有峰，别处什么都没有。** 这才是诊断，不是"发现了一个峰"。

### 健康：4.3 / 9.7 / 2.2 —— 误报底线

三个频率上全是个位数。**这一行是整张表的标尺** —— 没有它，你不知道 237 算不算大。

对比步骤 03：那时健康记录在 BPFI 处的幅值和内圈故障几乎一样（0.0093 vs 0.0124），会导致误判。**现在差了 24 倍。** 区别就在于带通滤波把那根冒充的机器谱线扔掉了。

### 但是先别急 —— 回头仔细看图 5b 的第一行

**健康那张图里最高的那根峰，也落在 160 Hz 附近，也就是 BPFI 线上。** 归一化之后它顶到了 1.0，看起来和内圈那根一样高。

只看图，你会以为健康轴承也有内圈故障。

**救命的是数字**：那根峰的峰背比是 **9.7**，内圈是 **237**。差 24 倍。

> 为什么归一化的图会这样？因为每个子图都被**除以了它自己的最大值**。健康记录整体是一片低矮的噪声，其中随便哪根稍高一点的毛刺，归一化后都会变成 1.0。**"最高的那根"和"突出的那根"是两回事。**
>
> 这就是为什么我一直在用峰背比而不是看图说话：它量的是"这根峰比它周围高多少"，不是"比全图最高的那根高多少"。**在一片噪声里，前者接近 1，后者可以是任何值。**

这一条要记住 —— 你以后画任何归一化的图都会遇到。**归一化让形状可比，但把"绝对显著性"抹掉了。图用来看形状，结论靠数字。**

### 滚珠：2.1 —— 失败，和预告的一样

BSF 处 2.1，健康记录在同一处是 2.2。**完全无法区分。**

图 4a 第四行已经解释过原因：滚珠故障的能量带很亮但**不闪**。有能量，没节拍；而包络谱量的正是节拍。

---
## 8. 两个值得知道的细节

### 细节一：内圈有边带，而且和理论不完全一致

内圈故障有个特点：坑跟着轴转，一会儿转进承载区、一会儿转出去，**撞击力道以轴频（29.5 Hz）起伏**。这种"调制的调制"在包络谱里表现为主峰两侧的**边带**（sidebands），间距等于轴频。

下面这格把边带量出来。**我要提醒你注意实测和教科书说法的出入** —— 结果在格子里。

In [ ]:
print("内圈：BPFI 周围的边带（教科书预期：间距 = 轴频 29.5 Hz）\n")
f, A = espec["Inner race"]
bpfi = inner.fault_freqs()["BPFI"]
sh = inner.n
for label, f0 in [("BPFI-2n", bpfi - 2*sh), ("BPFI-n", bpfi - sh), ("BPFI", bpfi),
                  ("BPFI+n", bpfi + sh), ("BPFI+2n", bpfi + 2*sh)]:
    r, fp = peak_ratio(f, A, f0)
    print(f"  {label:<9}{f0:>8.1f} Hz   峰背比 {r:>7.1f}")

print("\n外圈：BPFO 周围（教科书预期：外圈不动，力道恒定，不该有边带）\n")
f, A = espec["Outer race"]
bpfo = outer.fault_freqs()["BPFO"]
shn = outer.n
for label, f0 in [("BPFO-n", bpfo - shn), ("BPFO", bpfo), ("BPFO+n", bpfo + shn)]:
    r, fp = peak_ratio(f, A, f0)
    print(f"  {label:<9}{f0:>8.1f} Hz   峰背比 {r:>7.1f}")

**实测结果和教科书说法对不上，如实记下来：**

- 内圈的 ±1 倍轴频边带很弱（8.8 和 12.7），反而 ±2 倍轴频那一对更强（98.7 和 50.8）
- 外圈**也有**轴频边带（106 和 113），按教科书它不该有

我不打算给这个现象编一个解释。**这是一个我们没有完全搞清的观察**，报告里应该原样写出来，注明"与常见文献描述不完全一致，原因未进一步探究"。

> **这种处理方式本身就是分数。** 数据不支持教科书时，如实报告并标明未解释，远好过挑一个听起来合理的说法糊过去 —— 后者一旦被追问就塌了。

### 细节二：忘记给包络去均值会怎样

包络永远是正的（它是个模长），所以它的均值不为零。不减掉的话，频谱在 0 Hz 处会出现一根直流峰。

In [ ]:
e = envelope(outer.x[:N])
n = len(e)
f = np.fft.rfftfreq(n, 1 / FS)

for demean, label in [(True, "减了均值"), (False, "没减均值")]:
    ee = e - e.mean() if demean else e
    A = np.abs(np.fft.rfft(ee * np.hanning(n))) / n * 4
    a_dc = A[0]
    a_bpfo = A[np.abs(f - bpfo) <= 2.5].max()
    print(f"  {label}:  0 Hz 幅值 {a_dc:9.5f}   BPFO 幅值 {a_bpfo:9.5f}   "
          f"0Hz/BPFO = {a_dc/a_bpfo:6.2f}")

实测：不减均值时 0 Hz 的峰是 BPFO 峰的 **2.3 倍**。

常见教程会说"这根直流峰会把整张图压扁，你什么都看不见"。**实测没那么夸张** —— BPFO 峰仍有 43% 的高度，看得见。

但它确实有害：纵轴被 0 Hz 那根撑开，图里所有真实的峰都矮了一半；而且如果你写个"找最高峰"的自动化程序，它会永远返回 0 Hz。

**所以 `env - env.mean()` 这行照样不能漏，只是你现在知道漏了会怎样、有多严重** —— 而不是背一条"据说很可怕"的规矩。

---
## 9. 和步骤 04 比，这一节到底好在哪

| | 步骤 04（STFT 带内能量） | 步骤 05（带通 + 希尔伯特） |
|---|---|---|
| 外圈 BPFO | 601 | **635** |
| 内圈 BPFI | 222 | **237** |
| 滚珠 BSF | 2.3 | 2.1 |
| 频带边界 | 只能落在 187.5 Hz 的格子上 | **想选多窄就多窄** |
| 时间分辨率 | 被窗长绑死 | **原始的 1/12000 秒** |
| 参数 | 窗长、重叠，选错直接失效 | 只有频带 |

**数值上只好了一点点。** 这很重要，你应该知道 —— 步骤 04 那个"土办法"本质上已经是包络分析了，它并不差。

**真正的差别在最后三行**：希尔伯特方法**没有时频权衡这个包袱**。它直接在原始采样率上求包络，频带边界可以精确到 1 Hz。

**而这恰恰让最后一个问题变得无处可藏。**

## 那个问题

整节我都在用 `BAND = (2000, 4000)`。

**凭什么是这一段？**

- 我是从图 3a 看到"能量鼓包大概在这儿"，然后手指一划定的
- 步骤 04 的练习 2 里你已经看到：**换个频带，峰背比会变**
- 而且我在写步骤 04 时偷看过：**内圈的最佳频带和外圈不一样**

现在希尔伯特方法把频带边界的自由度完全交给了你 —— 无穷多种选法，每一种给出不同的结果。

**步骤 06 就是回答这个问题。** 那是整个项目里唯一一个没有标准答案、需要你自己拿主意并且论证的地方。

前面五步，任何人照着做都能做出来。**第六步做成什么样，才真正区分出人。**

---
## 10. 保存

In [ ]:
FIGURES.mkdir(exist_ok=True)
for fig, name in [(fig5a, "fig05a_envelope_construction.png"),
                  (fig5b, "fig05b_envelope_spectra.png")]:
    p = FIGURES / name
    fig.savefig(p, dpi=150, bbox_inches="tight")
    print("已保存:", p)

p = FIGURES / "table04_envelope_peak_ratios.txt"
with open(p, "w", encoding="utf-8") as fh:
    fh.write("Envelope spectrum, peak-to-background ratio\n")
    fh.write(f"Band {BAND[0]}-{BAND[1]} Hz, 4th-order Butterworth, zero phase, "
             f"{N} samples at {FS} Hz\n\n")
    fh.write(f"{'':<14}{'BPFO':>10}{'BPFI':>10}{'BSF':>10}    own fault\n")
    fh.write("-" * 60 + "\n")
    for s in signals:
        f, A = espec[s.name]
        ff = s.fault_freqs()
        vals = [peak_ratio(f, A, ff[k])[0] for k in ("BPFO", "BPFI", "BSF")]
        fh.write(f"{s.name:<14}" + "".join(f"{v:>10.1f}" for v in vals)
                 + f"    {TARGET[s.name]}\n")
    fh.write("\nThe healthy row is the false-alarm floor.\n")
    fh.write("Harmonics of BPFO in the outer-race record:\n")
    f, A = espec["Outer race"]
    for k in range(1, 6):
        fh.write(f"  {k} x BPFO = {k*bpfo:6.1f} Hz  ratio {peak_ratio(f, A, k*bpfo)[0]:7.1f}\n")
print("已保存:", p)

---
## 11. 自己动手

和上一节一样，**每格独立，直接运行**，改最上面那一行就能试别的值。

In [ ]:
# 练习 1：滤波器阶数的影响
# ------------------------------------------------
# 阶数越高，频带边缘切得越陡，但也越容易不稳定
ORDERS = [2, 4, 6, 8, 10]

print(f"{'阶数':>6}{'Outer':>10}{'Inner':>10}{'Ball':>10}{'Healthy':>10}")
print("-" * 48)
for od in ORDERS:
    row = []
    for s in signals[1:] + [signals[0]]:
        f, A = envelope_spectrum(s.x[:N], band=BAND)
        # 用指定阶数重算
        nyq = FS / 2
        b, a = butter(od, [BAND[0]/nyq, BAND[1]/nyq], btype="band")
        e = np.abs(hilbert(filtfilt(b, a, s.x[:N])))
        e = e - e.mean()
        nn = len(e)
        ff_ax = np.fft.rfftfreq(nn, 1/FS)
        AA = np.abs(np.fft.rfft(e * np.hanning(nn))) / nn * 4
        row.append(peak_ratio(ff_ax, AA, s.fault_freqs()[TARGET[s.name]])[0])
    print(f"{od:>6}" + "".join(f"{v:>10.1f}" for v in row))

print("\n阶数从 4 变到 10，结论变了吗？（结论对参数不敏感，是好事）")

In [ ]:
# 练习 2：录多久才够 —— 频率分辨率的代价
# ------------------------------------------------
DURATIONS = [0.5, 1, 2, 5, 10]     # 秒

print(f"{'时长(s)':>8}{'点数':>9}{'分辨率(Hz)':>12}{'Outer 峰背比':>14}{'实测峰':>9}")
print("-" * 55)
for d in DURATIONS:
    nn = int(d * FS)
    f, A = envelope_spectrum(outer.x[:nn], band=BAND)
    r, fp = peak_ratio(f, A, bpfo)
    print(f"{d:>8}{nn:>9}{FS/nn:>12.3f}{r:>14.1f}{fp:>9.1f}")

print("\n录 0.5 秒还能诊断吗？峰的位置准不准？")

In [ ]:
# 练习 3：看一眼步骤 06 要面对的问题
# ------------------------------------------------
# 系统扫一遍频带，看峰背比怎么变。这就是下一节的开场。
BANDS = [(500, 1500), (1000, 2000), (1500, 2500), (2000, 3000),
         (2500, 3500), (3000, 4000), (3500, 4500), (4000, 5000),
         (2000, 4000), (1000, 3000)]

print(f"{'频带':<16}{'Outer':>9}{'Inner':>9}{'Ball':>9}{'Healthy':>10}")
print("-" * 55)
for bd in BANDS:
    row = []
    for s in signals[1:] + [signals[0]]:
        f, A = envelope_spectrum(s.x[:N], band=bd)
        row.append(peak_ratio(f, A, s.fault_freqs()[TARGET[s.name]])[0])
    star = "  *" if bd == BAND else ""
    print(f"{str(bd):<16}" + "".join(f"{v:>9.1f}" for v in row[:3])
          + f"{row[3]:>10.1f}{star}")

print("\n(* = 本节用的频带)")
print("\n三个问题：")
print("  1. 外圈最好的频带是哪个？比 2000-4000 好多少？")
print("  2. 内圈最好的频带和外圈是同一个吗？")
print("  3. 健康记录在哪个频带刷出了最高值？那个值有多高？")
print("     —— 这个数决定了'多大的峰背比才算数'")

---
## 小结

| 做了 | 结论 |
|---|---|
| 人工信号验证（已知答案） | 直接 FFT 3.0，包络谱 31.8，峰精确在 105.90 Hz |
| 图 5a 三步构造 | 包络的鼓包与理论撞击时刻对齐 |
| 外圈 | **635**，五次谐波成列，误差 0.4% |
| 内圈 | **237**，交叉检验干净，误差 −0.25% |
| 健康（误报底线） | 4.3 / 9.7 / 2.2 |
| 滚珠 | 2.1 —— 失败，原因已在图 4a 看到 |
| 边带 | 与教科书说法不完全一致，**如实记录、不强行解释** |

**三种故障，两种诊断成功。** 外圈和内圈都是"只在理论预测的那一处有峰、别处没有"，并且和从轴承几何算出的频率吻合到 0.4% 以内。

**下一步（步骤 06）**：频带凭什么选 2000–4000？

这是整个项目里唯一一个需要你自己做判断的问题。做法是系统扫描、用峰背比打分、和步骤 04 的时频图交叉印证，最后给出一个**有论据的选择**。练习 3 已经把数据摆在你面前了。